# Unit IV Practical: Dimensionality Reduction – PCA, t-SNE and UMAP

**Aim:** To reduce high-dimensional data to a few dimensions using **PCA** while preserving maximum variance,
and to compare **PCA, t-SNE and UMAP** for visualizing high-dimensional data.

### Practicals covered
| Practical | Task | Dataset |
|---|---|---|
| 9 | Apply PCA to reduce dimensionality while preserving maximum variance | Wine (13 features) |
| 10 | Compare PCA, t-SNE and UMAP for visualizing high-dimensional data | Digits (64 features) |

### Syllabus covered
- Need for dimensionality reduction
- PCA – geometric intuition, explained variance, choosing the number of components
- Manifold learning overview
- t-SNE and UMAP (conceptual use)

> Run the cells from top to bottom. Each output is followed by a short note on **what to look for**.

## 0. Install UMAP (Google Colab)

PCA and t-SNE are inside scikit-learn. UMAP is a separate library, so we install it once.

In [ ]:
!pip install -q umap-learn

In [ ]:
# Step 1: Import libraries

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
import umap

print("Libraries imported successfully!")

# Practical 9 – Principal Component Analysis (PCA)

## 1. Why reduce dimensions?

- With many features, data **cannot be plotted** and is hard to understand.
- Many features are **correlated**, so they repeat the same information.
- In high dimensions, distances between points become less meaningful, which hurts clustering.

**PCA** finds new axes, called **principal components**, in the directions where the data **spreads out the most**.
Keeping only the first few components keeps most of the information (variance) with far fewer features.

## 2. Geometric intuition on a small 2D example

Before using a real dataset, look at 300 points with two **correlated** features.
PCA finds the direction of the longest spread (**PC1**) and the direction at a right angle to it (**PC2**).

In [ ]:
rng = np.random.default_rng(42)
x1 = rng.normal(0, 2, 300)
x2 = 0.8 * x1 + rng.normal(0, 0.7, 300)
toy = np.column_stack([x1, x2])

pca_toy = PCA(n_components=2).fit(toy)

plt.figure(figsize=(6, 6))
plt.scatter(toy[:, 0], toy[:, 1], alpha=0.4)
centre = toy.mean(axis=0)
for length, vector, name in zip(pca_toy.explained_variance_, pca_toy.components_, ["PC1", "PC2"]):
    end = centre + vector * 3 * np.sqrt(length)
    plt.annotate("", xy=end, xytext=centre, arrowprops=dict(arrowstyle="->", lw=3, color="red"))
    plt.text(end[0], end[1], name, fontsize=13, color="red")
plt.axis("equal")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("PCA finds the directions of largest spread")
plt.show()

print("Variance explained by PC1:", round(pca_toy.explained_variance_ratio_[0] * 100, 1), "%")
print("Variance explained by PC2:", round(pca_toy.explained_variance_ratio_[1] * 100, 1), "%")

**What to look for:** PC1 points along the long direction of the cloud and explains most of the variance.
If we kept **only PC1**, we would describe each point with one number instead of two and lose very little.
This is dimensionality reduction.

## 3. Load the Wine dataset

178 wines from 3 cultivars, each described by **13 chemical measurements**.
The cultivar label is kept aside and used **only for colouring plots**. PCA never sees it.

In [ ]:
wine = load_wine(as_frame=True)
X_wine = wine.data
cultivar = wine.target

print("Dataset shape:", X_wine.shape)
print("Features:", list(X_wine.columns))

## 4. Standardize the data

PCA looks for directions of large variance. A feature measured in large units (for example `proline`,
in the hundreds) would look important only because of its units. Standardizing gives every feature
mean 0 and standard deviation 1.

In [ ]:
X_wine_scaled = StandardScaler().fit_transform(X_wine)

print("Variance of each raw feature (first 5):", X_wine.var().round(1).values[:5])
print("Variance after standardizing (first 5):", X_wine_scaled.var(axis=0).round(2)[:5])

## 5. Fit PCA and look at the explained variance

We fit PCA with **all 13 components** first, to see how much variance each one keeps.

In [ ]:
pca_full = PCA().fit(X_wine_scaled)
ratio = pca_full.explained_variance_ratio_ * 100
cumulative = np.cumsum(ratio)

table = pd.DataFrame({
    "Component": [f"PC{i}" for i in range(1, 14)],
    "Variance explained (%)": ratio.round(1),
    "Cumulative (%)": cumulative.round(1),
})
table

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(range(1, 14), ratio, label="Each component")
plt.plot(range(1, 14), cumulative, marker="o", color="red", label="Cumulative")
plt.axhline(90, linestyle="--", color="grey", label="90 %")
plt.axhline(95, linestyle=":", color="black", label="95 %")
plt.xticks(range(1, 14))
plt.xlabel("Principal component")
plt.ylabel("Variance explained (%)")
plt.title("Scree plot – Wine dataset")
plt.legend()
plt.show()

for target in (80, 90, 95):
    k = int(np.argmax(cumulative >= target) + 1)
    print(f"Components needed for {target}% of the variance: {k} of 13")

**What to look for:** the first components are tall and the later ones are small. Only **5 of the 13** components
keep 80 % of the variance and **8** keep 90 %; each of the last components adds less than 2 %. So the last few
components can be dropped with little loss. This is what "preserving maximum variance" means.

## 6. Reduce to 2 components and visualize

In [ ]:
pca_2 = PCA(n_components=2)
wine_2d = pca_2.fit_transform(X_wine_scaled)

plt.figure(figsize=(7, 5))
for c in sorted(cultivar.unique()):
    plt.scatter(wine_2d[cultivar == c, 0], wine_2d[cultivar == c, 1], label=f"Cultivar {c}")
plt.xlabel(f"PC1 ({pca_2.explained_variance_ratio_[0] * 100:.1f} % variance)")
plt.ylabel(f"PC2 ({pca_2.explained_variance_ratio_[1] * 100:.1f} % variance)")
plt.title("13 features reduced to 2 with PCA")
plt.legend()
plt.show()

**What to look for:** even with only 2 of 13 dimensions, the three cultivars form three visible groups.
PCA did this without using the labels.

## 7. How much is lost? Reconstruction error

PCA can map the reduced data **back** to 13 features. The difference between the original and the rebuilt
data is the **reconstruction error**. More components → smaller error.

In [ ]:
errors = []
for k in range(1, 14):
    p = PCA(n_components=k).fit(X_wine_scaled)
    rebuilt = p.inverse_transform(p.transform(X_wine_scaled))
    errors.append(np.mean((X_wine_scaled - rebuilt) ** 2))

plt.figure(figsize=(7, 4))
plt.plot(range(1, 14), errors, marker="o")
plt.xticks(range(1, 14))
plt.xlabel("Number of components kept")
plt.ylabel("Mean squared reconstruction error")
plt.title("Keeping more components loses less information")
plt.show()

**What to look for:** the error falls quickly at first and then slowly, and reaches **0** when all 13
components are kept (nothing is thrown away).

## 8. What does each component mean? (Loadings)

Each principal component is a weighted mix of the original features. A large weight (positive or negative)
means the feature contributes strongly to that component.

In [ ]:
loadings = pd.DataFrame(pca_2.components_.T, index=X_wine.columns, columns=["PC1", "PC2"])
loadings.reindex(loadings["PC1"].abs().sort_values(ascending=False).index).round(2)

**What to look for:** the features at the top of the table drive PC1. Features with similar weights move
together, which is why PCA can replace them with one component.

### Practical 9 – Result
PCA reduced 13 correlated features to a few components. A small number of components preserved most of
the variance, and 2 components were enough to see the cultivar structure.

# Practical 10 – Comparing PCA, t-SNE and UMAP

## 9. Load the Digits dataset

1,797 images of handwritten digits. Each image is **8 × 8 = 64 pixels**, so each image is a point in
**64 dimensions**. We want to see this data in 2D.

In [ ]:
digits = load_digits()
X_digits = digits.data
digit_label = digits.target          # used only for colouring

print("Dataset shape:", X_digits.shape)

fig, axes = plt.subplots(1, 10, figsize=(10, 1.5))
for d, ax in enumerate(axes):
    ax.imshow(digits.images[np.where(digit_label == d)[0][0]], cmap="gray_r")
    ax.set_title(str(d))
    ax.axis("off")
plt.show()

X_digits_scaled = StandardScaler().fit_transform(X_digits)

## 10. Three methods, one idea each

| Method | Type | Idea in one line |
|---|---|---|
| **PCA** | Linear | Projects the data flat onto the directions of largest variance |
| **t-SNE** | Non-linear (manifold learning) | Keeps points that are neighbours in 64D as neighbours in 2D |
| **UMAP** | Non-linear (manifold learning) | Like t-SNE, keeps neighbours together; usually faster and keeps more of the overall layout |

PCA can only **flatten**. t-SNE and UMAP can **unfold** curved structure, because they care about each point's
neighbours rather than straight-line directions.

## 11. Run all three and time them

In [ ]:
embeddings = {}
times = {}

start = time.time()
embeddings["PCA"] = PCA(n_components=2, random_state=42).fit_transform(X_digits_scaled)
times["PCA"] = time.time() - start

start = time.time()
embeddings["t-SNE"] = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_digits_scaled)
times["t-SNE"] = time.time() - start

start = time.time()
embeddings["UMAP"] = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                               random_state=42).fit_transform(X_digits_scaled)
times["UMAP"] = time.time() - start

for name, t in times.items():
    print(f"{name:6s} took {t:6.2f} seconds")

**What to look for:** PCA is almost instant. t-SNE and UMAP take longer because they work with neighbours
of every point. (Times depend on your computer.)

## 12. Visualize side by side

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (name, emb) in zip(axes, embeddings.items()):
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=digit_label, cmap="tab10", s=6)
    for d in range(10):
        cx, cy = np.median(emb[digit_label == d], axis=0)
        ax.text(cx, cy, str(d), fontsize=16, weight="bold",
                bbox=dict(facecolor="white", alpha=0.7, boxstyle="round"))
    ax.set_title(f"{name}  ({times[name]:.1f} s)")
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("64-dimensional digits shown in 2D (colours = true digit, not used by the methods)")
plt.show()

**What to look for:**
- **PCA:** the digits overlap heavily in the middle. Two straight directions cannot separate 10 digits.
- **t-SNE:** each digit forms its own tight island. Neighbours in 64D stay neighbours in 2D.
- **UMAP:** also clear islands, usually more compact, produced faster than t-SNE.

**Careful:** in t-SNE plots, the **size of an island** and the **gap between islands** do not have a reliable
meaning. Read t-SNE for "which points belong together", not for distances between groups.

## 13. Measure it: how well are neighbours kept?

`trustworthiness` (from scikit-learn) checks whether points that are neighbours in the 2D picture were
also neighbours in the original 64D data. It is between 0 and 1; higher is better.

In [ ]:
rows = []
for name, emb in embeddings.items():
    rows.append({
        "Method": name,
        "Trustworthiness (k=10)": round(trustworthiness(X_digits_scaled, emb, n_neighbors=10), 3),
        "Time (s)": round(times[name], 2),
    })
pd.DataFrame(rows)

**What to look for:** t-SNE and UMAP keep local neighbourhoods much better than PCA. PCA is the fastest
and is still the best choice when you need a simple, repeatable, **linear** reduction (for example, before
clustering or as input to a model).

## 14. Effect of t-SNE's main setting: perplexity

**Perplexity** is roughly how many neighbours t-SNE pays attention to for each point.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, perp in zip(axes, [5, 30, 50]):
    emb = TSNE(n_components=2, perplexity=perp, random_state=42).fit_transform(X_digits_scaled)
    ax.scatter(emb[:, 0], emb[:, 1], c=digit_label, cmap="tab10", s=5)
    ax.set_title(f"t-SNE, perplexity = {perp}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

**What to look for:** with a **small perplexity** (5) each island is loose and patchy, and some digits split into
separate pieces (look at the 1s). With 30 or 50 the islands become tighter and smoother. The picture depends on this
setting, so always try a few values before drawing conclusions.

## 15. Quick Comparison

| | PCA | t-SNE | UMAP |
|---|---|---|---|
| Type | Linear | Non-linear | Non-linear |
| What it keeps | Directions of maximum variance | Local neighbourhoods | Local neighbourhoods and more of the overall layout |
| Speed | Very fast | Slow | Faster than t-SNE |
| Same result every run? | Yes | Changes with random seed | Changes with random seed |
| Can transform new points? | Yes | No (must re-run) | Yes |
| Main use | Compression, preprocessing, first look | Visualization only | Visualization and preprocessing |

## Conclusion

PCA reduced the 13 wine features to a few components while preserving most of the variance,
and showed how much information each component carries. On the 64-dimensional digits, PCA's straight-line
projection left the digits overlapping, while t-SNE and UMAP **unfolded** the data into clear groups by
keeping neighbours together. Use **PCA** to compress and preprocess, and **t-SNE / UMAP** to look at
complex high-dimensional data.